# Use only flight information baseline


In [22]:
from pathlib import Path
import pandas as pd

flights = pd.read_csv(Path("Data") / "flights_clean.csv")

delay = flights[
    (flights["Cancelled"] == 0)
    & (flights["Diverted"] == 0)
    & flights["ArrDel15"].isin([0, 1])
].copy()

In [23]:
from sklearn.preprocessing import OneHotEncoder

features = ["Reporting_Airline", "Dest", "Month", "DayOfWeek", "DepHour"]
train = delay[delay["Year"].between(2022, 2024)]
val = delay[delay["Year"] == 2025]

encoder = OneHotEncoder(handle_unknown="ignore")
X_train = encoder.fit_transform(train[features])
X_val = encoder.transform(val[features])

y_train = train["ArrDel15"].astype(int)
y_val = val["ArrDel15"].astype(int)

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for weight in [None, "balanced"]:
    model = LogisticRegression(class_weight=weight, max_iter=500)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    print("class_weight =", weight)
    print("Accuracy:", round(accuracy_score(y_val, pred), 3))
    print("Precision:", round(precision_score(y_val, pred, zero_division=0), 3))
    print("Recall:", round(recall_score(y_val, pred), 3))
    print("F1:", round(f1_score(y_val, pred), 3))

class_weight = None
Accuracy: 0.758
Precision: 0.49
Recall: 0.026
F1: 0.048
class_weight = balanced
Accuracy: 0.625
Precision: 0.34
Recall: 0.588
F1: 0.431


# Adding Historical weather information & 6 hour forecast & Decision Tree

In [25]:
delay_sample = pd.read_csv(Path("Data") / "delay_model_sample.csv")
weather = pd.read_csv(Path("Data") / "booking_weather_features.csv")

# Use BOS departure and the historical window from 1999 to 2021
weather_cols = [
    "Origin_1999_Wind20",
    "Origin_1999_VisBelow3",
    "Origin_1999_CeilingBelow1000",
    "Origin_1999_Snow",
    "Origin_1999_Thunder",
]

# merge on FlightRowID to get the weather for each flight
data = delay_sample.merge(
    weather[["FlightRowID"] + weather_cols],
    on="FlightRowID",
    how="left",
    validate="one_to_one",
)

# Check the proportion of successful matches for each weather feature, 1.0 means 100%
print(data[weather_cols].notna().mean().round(3))

Origin_1999_Wind20              1.0
Origin_1999_VisBelow3           1.0
Origin_1999_CeilingBelow1000    1.0
Origin_1999_Snow                1.0
Origin_1999_Thunder             1.0
dtype: float64


## Adding weather information 

In [26]:
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler

train = data[data["Year"].between(2022, 2024)]
val = data[data["Year"] == 2025]

y_train = train["ArrDel15"].astype(int)
y_val = val["ArrDel15"].astype(int)

# use feature from  baseline and OneHotEncoder
X_train = encoder.fit_transform(train[features])
X_val = encoder.transform(val[features])

# replace missing weather values with the median from the training set. Missing and "occurrence ratio of 0" have different meanings
weather_median = train[weather_cols].median()
train_weather = train[weather_cols].fillna(weather_median)
val_weather = val[weather_cols].fillna(weather_median)

# The occurrence frequencies of the five weather conditions vary significantly, so standardize using the mean and standard deviation from the training set
scaler = StandardScaler()
train_weather = scaler.fit_transform(train_weather)
val_weather = scaler.transform(val_weather)

# add the five weather columns to the right of the baseline features, keeping each row as the original flight
X_train_weather = hstack([X_train, train_weather], format="csr")
X_val_weather = hstack([X_val, val_weather], format="csr")

## Adding six hour weather information 

In [27]:
taf_cols = ["TAFWindKt_6h", "TAFVisBelow3_6h"]

taf = pd.read_csv(
    Path("Data") / "bos_taf_6h_features.csv",
    usecols=["FlightRowID"] + taf_cols,
).set_index("FlightRowID", verify_integrity=True)

# take forecasts in the order of the existing training and validation flights, ensuring each row corresponds to the same flight
train_taf = taf.reindex(train["FlightRowID"])[taf_cols]
val_taf = taf.reindex(val["FlightRowID"])[taf_cols]

# how many flights have forecasts for each column
pd.DataFrame({
    "Train coverage": train_taf.notna().mean(),
    "Validation coverage": val_taf.notna().mean(),
}).round(3)


# keep the same flights and fill missing values with the median from the training set
taf_median = train_taf.median()
taf_scaler = StandardScaler()

train_taf_scaled = taf_scaler.fit_transform(train_taf.fillna(taf_median))
val_taf_scaled = taf_scaler.transform(val_taf.fillna(taf_median))

# X_train_weather already contains the schedule + five historical weather features
X_train_6h = hstack([X_train_weather, train_taf_scaled], format="csr")
X_val_6h = hstack([X_val_weather, val_taf_scaled], format="csr")

C:\Users\Owner\AppData\Local\Temp\ipykernel_30792\3227811842.py:6: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  ).set_index("FlightRowID", verify_integrity=True)


In [28]:
experiments = [
    ("Schedule only", X_train, X_val),
    ("Schedule + BOS climate", X_train_weather, X_val_weather),
    ("Schedule + BOS climate + 6h TAF", X_train_6h, X_val_6h),
]

results = []
for name, train_x, val_x in experiments:
    model = LogisticRegression(class_weight="balanced", max_iter=500)
    model.fit(train_x, y_train)
    pred = model.predict(val_x)

    results.append({
        "Features": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
    })

pd.DataFrame(results).round(4)

,Features,Accuracy,Precision,Recall,F1
0,Schedule only,0.6245,0.3399,0.5884,0.4309
1,Schedule + BOS climate,0.6314,0.3446,0.5834,0.4333
2,Schedule + BOS climate + 6h TAF,0.6344,0.3492,0.5943,0.4399


In [29]:
#  Use logistic regression for delay prediction with 6-hour features
delay_6h_model = LogisticRegression(class_weight="balanced", max_iter=500)
delay_6h_model.fit(X_train_6h, y_train)
delay_scores = delay_6h_model.predict_proba(X_val_6h)[:, 1]

In [30]:
#  Evaluate delay prediction thresholds on the 2025 validation set
delay_results = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (delay_scores > threshold).astype(int)
    delay_results.append({
        "Threshold": threshold,
        "Alerts": int(pred.sum()),
        "Delays caught": int(((pred == 1) & (y_val == 1)).sum()),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
    })

delay_threshold_table = pd.DataFrame(delay_results)
best_row = delay_threshold_table["F1"].idxmax()
best_delay_threshold = delay_threshold_table.loc[best_row, "Threshold"]
print("Selected delay threshold (2025 validation):", best_delay_threshold)
delay_threshold_table.round(4)

Selected delay threshold (2025 validation): 0.4


,Threshold,Alerts,Delays caught,Precision,Recall,F1
0,0.3,115774,31431,0.2715,0.9289,0.4202
1,0.4,87292,26897,0.3081,0.7949,0.4441
2,0.5,57597,20110,0.3492,0.5943,0.4399
3,0.6,31239,12418,0.3975,0.3670,0.3816
4,0.7,12179,5485,0.4504,0.1621,0.2384
5,0.8,2023,1106,0.5467,0.0327,0.0617
6,0.9,33,23,0.6970,0.0007,0.0014


## Decision Tree

In [31]:
from sklearn.tree import DecisionTreeClassifier

tree_results = []

# Use the same flights and features, only comparing the maximum depth of the tree
for depth in [3, 5, 8]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42,
    )
    tree.fit(X_train_6h, y_train)
    pred = tree.predict(X_val_6h)

    tree_results.append({
        "Max depth": depth,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
    })

pd.DataFrame(tree_results).round(4)

,Max depth,Accuracy,Precision,Recall,F1
0,3,0.6508,0.3323,0.4418,0.3793
1,5,0.6550,0.3392,0.4517,0.3875
2,8,0.6741,0.3543,0.4248,0.3864


### Not good enough, use logistic regression for final prediction

# Predict Delay

In [32]:
# data already contains flights and historical weather, now join the two TAF columns by FlightRowID
final_data = data.join(taf[taf_cols], on="FlightRowID", validate="one_to_one")

final_train = final_data[final_data["Year"].between(2022, 2025)].copy()
test = final_data[final_data["Year"] == 2026].copy()

print("Training flights:", len(final_train))
print("Test flights:", len(test))
print("Test dates:", test["FlightDate"].min(), "to", test["FlightDate"].max())

Training flights: 550318
Test flights: 80808
Test dates: 2026-01-01 to 2026-07-31


In [33]:
# schedule features: for 2026, only use the encoding learned during training
final_encoder = OneHotEncoder(handle_unknown="ignore")
schedule_train = final_encoder.fit_transform(final_train[features])
schedule_test = final_encoder.transform(test[features])

# historical weather and TAF are both numeric columns and can be processed column by column
numeric_cols = weather_cols + taf_cols
medians = final_train[numeric_cols].median()

final_scaler = StandardScaler()
weather_train = final_scaler.fit_transform(final_train[numeric_cols].fillna(medians))
weather_test = final_scaler.transform(test[numeric_cols].fillna(medians))

X_final_train = hstack([schedule_train, weather_train], format="csr")
X_test = hstack([schedule_test, weather_test], format="csr")

In [34]:
from sklearn.metrics import confusion_matrix

final_model = LogisticRegression(class_weight="balanced", max_iter=500)
final_model.fit(X_final_train, final_train["ArrDel15"].astype(int))

y_test = test["ArrDel15"].astype(int)
test_pred = final_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))

pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=[0, 1]),
    index=["Actual: no delay ≥15 min", "Actual: delay ≥15 min"],
    columns=["Predicted: no delay ≥15 min", "Predicted: delay ≥15 min"],
)

Accuracy: 0.6281
Precision: 0.3541
Recall: 0.6386
F1: 0.4556


,Predicted: no delay ≥15 min,Predicted: delay ≥15 min
Actual: no delay ≥15 min,38178,22937
Actual: delay ≥15 min,7118,12575


In [35]:
# Use the same final model to compare the default threshold and the threshold selected on 2025 validation
delay_test_scores = final_model.predict_proba(X_test)[:, 1]
delay_test_pred = (delay_test_scores > best_delay_threshold).astype(int)

delay_test_results = []
for rule, threshold, pred in [
    ("Default", 0.5, test_pred),
    ("Selected on 2025 validation", best_delay_threshold, delay_test_pred),
]:
    delay_test_results.append({
        "Rule": rule,
        "Threshold": threshold,
        "Alerts": int(pred.sum()),
        "Delays caught": int(((pred == 1) & (y_test == 1)).sum()),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    })

pd.DataFrame(delay_test_results).round(4)

,Rule,Threshold,Alerts,Delays caught,Accuracy,Precision,Recall,F1
0,Default,0.5,35512,12575,0.6281,0.3541,0.6386,0.4556
1,Selected on 2025 validation,0.4,52117,16030,0.5081,0.3076,0.8140,0.4465


# Cancelation

In [36]:
cancel = pd.read_csv(Path("Data") / "cancel_model_sample.csv")

# use the same time split, first look at the training and validation sets
cancel_train = cancel[cancel["Year"].between(2022, 2024)]
cancel_val = cancel[cancel["Year"] == 2025]

for name, sample in [("Train", cancel_train), ("Validation", cancel_val)]:
    print(name, len(sample), "flights")
    print("Cancelled:", int(sample["Cancelled"].sum()))
    print("Cancellation rate:", round(sample["Cancelled"].mean() * 100, 2), "%")

Train 420483 flights
Cancelled: 9117
Cancellation rate: 2.17 %
Validation 142859 flights
Cancelled: 2395
Cancellation rate: 1.68 %


In [37]:
# reuse the five features from the delay model: airline, destination, month, day of week, departure hour
cancel_encoder = OneHotEncoder(handle_unknown="ignore")

X_cancel_train = cancel_encoder.fit_transform(cancel_train[features])
X_cancel_val = cancel_encoder.transform(cancel_val[features])

# the positive class is a cancelled flight: 1 = cancelled, 0 = not cancelled
y_cancel_train = cancel_train["Cancelled"].astype(int)
y_cancel_val = cancel_val["Cancelled"].astype(int)

In [38]:
cancel_results = []

for weight in [None, "balanced"]:
    model = LogisticRegression(class_weight=weight, max_iter=500)
    model.fit(X_cancel_train, y_cancel_train)
    pred = model.predict(X_cancel_val)

    cancel_results.append({
        "Class weight": "Default" if weight is None else "Balanced",
        "Predicted cancellations": int(pred.sum()),
        "Accuracy": accuracy_score(y_cancel_val, pred),
        "Precision": precision_score(y_cancel_val, pred, zero_division=0),
        "Recall": recall_score(y_cancel_val, pred, zero_division=0),
        "F1": f1_score(y_cancel_val, pred, zero_division=0),
    })

pd.DataFrame(cancel_results).round(4)

,Class weight,Predicted cancellations,Accuracy,Precision,Recall,F1
0,Default,0,0.9832,0.000,0.0000,0.0000
1,Balanced,50613,0.6481,0.027,0.5704,0.0515


In [39]:
# take forecasts in the order of the cancellation task flights, ensuring the weather matrix and schedule matrix correspond row by row
cancel_taf_train = taf[taf_cols].reindex(cancel_train["FlightRowID"])
cancel_taf_val = taf[taf_cols].reindex(cancel_val["FlightRowID"])

# before filling missing values, check how many percent flights each forecast feature covers
pd.DataFrame({
    "Train coverage": cancel_taf_train.notna().mean(),
    "Validation coverage": cancel_taf_val.notna().mean(),
}).round(3)

,Train coverage,Validation coverage
TAFWindKt_6h,0.999,0.999
TAFVisBelow3_6h,0.999,0.999


In [40]:
# fill missing values with the median from the cancellation task training set, keeping all validation flights
cancel_taf_median = cancel_taf_train.median()
train_weather = cancel_taf_train.fillna(cancel_taf_median)
val_weather = cancel_taf_val.fillna(cancel_taf_median)

# learn the standardization parameters from the training set, then apply them to the validation set
cancel_taf_scaler = StandardScaler()
train_weather = cancel_taf_scaler.fit_transform(train_weather)
val_weather = cancel_taf_scaler.transform(val_weather)

X_cancel_train_6h = hstack([X_cancel_train, train_weather], format="csr")
X_cancel_val_6h = hstack([X_cancel_val, val_weather], format="csr")

In [41]:
cancel_experiments = [
    ("Schedule only", X_cancel_train, X_cancel_val),
    ("Schedule + 6h BOS TAF", X_cancel_train_6h, X_cancel_val_6h),
]

cancel_weather_results = []

for name, train_x, val_x in cancel_experiments:
    model = LogisticRegression(class_weight="balanced", max_iter=500)
    model.fit(train_x, y_cancel_train)
    pred = model.predict(val_x)

    cancel_weather_results.append({
        "Features": name,
        "Alerts": int(pred.sum()),
        "Cancellations caught": int(((pred == 1) & (y_cancel_val == 1)).sum()),
        "Precision": precision_score(y_cancel_val, pred, zero_division=0),
        "Recall": recall_score(y_cancel_val, pred, zero_division=0),
        "F1": f1_score(y_cancel_val, pred, zero_division=0),
    })

pd.DataFrame(cancel_weather_results).round(4)

,Features,Alerts,Cancellations caught,Precision,Recall,F1
0,Schedule only,50613,1366,0.027,0.5704,0.0515
1,Schedule + 6h BOS TAF,45111,1397,0.031,0.5833,0.0588


In [42]:
cancel_6h_model = LogisticRegression(class_weight="balanced", max_iter=500)
cancel_6h_model.fit(X_cancel_train_6h, y_cancel_train)

# the first column corresponds to label 1, i.e., the model score for "cancelled"
cancel_scores = cancel_6h_model.predict_proba(X_cancel_val_6h)[:, 1]

In [43]:
threshold_results = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    # the model is trained only once, and only the alert threshold changes in each iteration
    pred = (cancel_scores > threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Alerts": int(pred.sum()),
        "Cancellations caught": int(((pred == 1) & (y_cancel_val == 1)).sum()),
        "Precision": precision_score(y_cancel_val, pred, zero_division=0),
        "Recall": recall_score(y_cancel_val, pred),
        "F1": f1_score(y_cancel_val, pred),
    })

threshold_table = pd.DataFrame(threshold_results)
threshold_table.round(4)

,Threshold,Alerts,Cancellations caught,Precision,Recall,F1
0,0.5,45111,1397,0.0310,0.5833,0.0588
1,0.6,25592,1029,0.0402,0.4296,0.0735
2,0.7,11390,620,0.0544,0.2589,0.0900
3,0.8,3344,285,0.0852,0.1190,0.0993
4,0.9,391,97,0.2481,0.0405,0.0696


# Decision Tree

In [44]:
#  Evaluate cancellation prediction thresholds on the 2025 validation set
cancel_tree_results = []

for depth in [3, 5, 8]:
    cancel_tree = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    cancel_tree.fit(X_cancel_train_6h, y_cancel_train)
    tree_scores = cancel_tree.predict_proba(X_cancel_val_6h)[:, 1]

    for threshold in threshold_table["Threshold"]:
        # Only alert when the cancellation score is strictly above the threshold
        pred = (tree_scores > threshold).astype(int)
        cancel_tree_results.append({
            "Max depth": depth,
            "Threshold": threshold,
            "Alerts": int(pred.sum()),
            "Cancellations caught": int(((pred == 1) & (y_cancel_val == 1)).sum()),
            "Precision": precision_score(y_cancel_val, pred, zero_division=0),
            "Recall": recall_score(y_cancel_val, pred),
            "F1": f1_score(y_cancel_val, pred),
        })

cancel_tree_table = pd.DataFrame(cancel_tree_results)
cancel_tree_table.round(4)

,Max depth,Threshold,Alerts,Cancellations caught,Precision,Recall,F1
0,3,0.5,5692,354,0.0622,0.1478,0.0875
1,3,0.6,3876,318,0.0820,0.1328,0.1014
2,3,0.7,3132,292,0.0932,0.1219,0.1057
3,3,0.8,3132,292,0.0932,0.1219,0.1057
4,3,0.9,0,0,0.0000,0.0000,0.0000
5,5,0.5,37470,1177,0.0314,0.4914,0.0590
6,5,0.6,5014,344,0.0686,0.1436,0.0929
7,5,0.7,2968,186,0.0627,0.0777,0.0694
8,5,0.8,2290,161,0.0703,0.0672,0.0687
9,5,0.9,1271,143,0.1125,0.0597,0.0780


In [45]:
# Find the row with the highest F1 for each model on the 2025 validation set, using unrounded values
lr_best = threshold_table.loc[threshold_table["F1"].idxmax()]
tree_best = cancel_tree_table.loc[cancel_tree_table["F1"].idxmax()]

best_cancel_tree_depth = int(tree_best["Max depth"])
best_cancel_tree_threshold = tree_best["Threshold"]
print("Selected tree depth:", best_cancel_tree_depth)

val_comparison = pd.DataFrame([lr_best, tree_best], index=["Logistic Regression", "Decision Tree"])
val_comparison[["Threshold", "Alerts", "Cancellations caught", "Precision", "Recall", "F1"]].round(4)

Selected tree depth: 3


,Threshold,Alerts,Cancellations caught,Precision,Recall,F1
Logistic Regression,0.8,3344.0,285.0,0.0852,0.1190,0.0993
Decision Tree,0.7,3132.0,292.0,0.0932,0.1219,0.1057


In [46]:
# select the threshold with the highest F1 from the 2025 validation results
best_threshold = threshold_table.loc[threshold_table["F1"].idxmax(), "Threshold"]
print("Selected threshold:", best_threshold)

# join the two BOS forecast features by flight number
cancel_data = cancel.join(taf[taf_cols], on="FlightRowID", validate="one_to_one")

cancel_final_train = cancel_data[cancel_data["Year"].between(2022, 2025)].copy()
cancel_test = cancel_data[cancel_data["Year"] == 2026].copy()

Selected threshold: 0.8


In [47]:
# reuse the five schedule features from the previous features
cancel_final_encoder = OneHotEncoder(handle_unknown="ignore")
schedule_train = cancel_final_encoder.fit_transform(cancel_final_train[features])
schedule_test = cancel_final_encoder.transform(cancel_test[features])

# reuse the two weather features from the previous taf_cols
taf_medians = cancel_final_train[taf_cols].median()
taf_train = cancel_final_train[taf_cols].fillna(taf_medians)
taf_test = cancel_test[taf_cols].fillna(taf_medians)

cancel_final_scaler = StandardScaler()
taf_train = cancel_final_scaler.fit_transform(taf_train)
taf_test = cancel_final_scaler.transform(taf_test)

X_cancel_final = hstack([schedule_train, taf_train], format="csr")
X_cancel_test = hstack([schedule_test, taf_test], format="csr")

In [48]:
cancel_final_model = LogisticRegression(class_weight="balanced", max_iter=500)
cancel_final_model.fit(X_cancel_final, cancel_final_train["Cancelled"].astype(int))

y_cancel_test = cancel_test["Cancelled"].astype(int)
test_scores = cancel_final_model.predict_proba(X_cancel_test)[:, 1]

# keep the same rule as Cell 20: only send an alert if the score is strictly greater than the selected threshold
cancel_test_pred = (test_scores > best_threshold).astype(int)

print("Test flights:", len(y_cancel_test))
print("Actual cancellations:", int(y_cancel_test.sum()))
print("Alerts:", int(cancel_test_pred.sum()))
print("Accuracy:", round(accuracy_score(y_cancel_test, cancel_test_pred), 4))
print("Precision:", round(precision_score(y_cancel_test, cancel_test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_cancel_test, cancel_test_pred), 4))
print("F1:", round(f1_score(y_cancel_test, cancel_test_pred), 4))

Test flights: 84109
Actual cancellations: 3064
Alerts: 3039
Accuracy: 0.9465
Precision: 0.2636
Recall: 0.2614
F1: 0.2625


In [49]:
pd.DataFrame(
    confusion_matrix(y_cancel_test, cancel_test_pred, labels=[0, 1]),
    index=["Actual: operated", "Actual: cancelled"],
    columns=["Predicted: operated", "Predicted: cancelled"],
)

,Predicted: operated,Predicted: cancelled
Actual: operated,78807,2238
Actual: cancelled,2263,801


In [50]:
# Fit the final cancellation tree and apply the threshold selected on 2025 validation
cancel_final_tree = DecisionTreeClassifier(
    max_depth=best_cancel_tree_depth,
    class_weight="balanced",
    random_state=42,
)
cancel_final_tree.fit(X_cancel_final, cancel_final_train["Cancelled"].astype(int))

# Apply the threshold selected on 2025 validation to the 2026 flights
tree_test_scores = cancel_final_tree.predict_proba(X_cancel_test)[:, 1]
cancel_tree_test_pred = (tree_test_scores > best_cancel_tree_threshold).astype(int)

In [51]:
# Evaluate cancellation prediction thresholds on the 2026 test set
cancel_test_results = []
for name, pred in [("Logistic Regression", cancel_test_pred), ("Decision Tree", cancel_tree_test_pred)]:
    cancel_test_results.append({
        "Model": name,
        "Alerts": int(pred.sum()),
        "Cancellations caught": int(((pred == 1) & (y_cancel_test == 1)).sum()),
        "Precision": precision_score(y_cancel_test, pred, zero_division=0),
        "Recall": recall_score(y_cancel_test, pred),
        "F1": f1_score(y_cancel_test, pred),
    })

pd.DataFrame(cancel_test_results).round(4)

,Model,Alerts,Cancellations caught,Precision,Recall,F1
0,Logistic Regression,3039,801,0.2636,0.2614,0.2625
1,Decision Tree,3633,713,0.1963,0.2327,0.2129


Modeling Methods

We modeled arrival delays of at least 15 minutes and flight cancellations as separate binary classification tasks. We began with airline, destination, month, day of week, and scheduled departure hour because these features are available at booking. For arrival delays, we added historical BOS weather frequencies to represent typical conditions for the departure month and hour. For both tasks, we tested whether forecast wind speed and visibility available six hours before scheduled departure provided additional information.

We trained on 2022–2024 flights and used 2025 for model selection. We evaluated precision, recall, and F1 because cancellations were rare and accuracy alone rewarded models that predicted every flight would operate. For arrival delays, class-weighted logistic regression achieved a validation F1 of 0.4399, compared with 0.3875 for the best decision tree tested. For cancellation logistic regression, we selected a threshold of 0.8 because it achieved the highest validation F1 among the five thresholds tested. We then retrained on 2022–2025 data and evaluated the delay and cancellation logistic regression models on January–July 2026 flights. For the cancellation decision tree, we compared depths of 3, 5, and 8 on the same 2025 flights and used the same five alert thresholds. We selected the depth and threshold together by validation F1.

Results & Interpretation

For arrival delays, validation F1 increased from 0.4309 with schedule features alone to 0.4333 with historical weather and 0.4399 after adding the six-hour forecast. At the original 0.5 threshold, the model identified 12,575 of 19,693 delayed flights on the 2026 test set, achieving 63.86% recall, 35.41% precision, and an F1 of 0.4556. In a follow-up experiment, we selected a delay threshold by 2025 validation F1. The delay test table compares this threshold with 0.5 on the same 2026 flights.

For cancellations, adding the forecast at a fixed threshold of 0.5 produced 5,502 fewer alerts while identifying 31 additional cancellations on the validation set. At the selected threshold of 0.8, logistic regression identified 801 of 3,064 cancellations on the test set, with 26.36% precision, 26.14% recall, and an F1 of 0.2625. Flagged flights had a cancellation rate about seven times the overall test rate, although the model missed most cancellations. The data record the final cancellation status but do not indicate when each cancellation was announced. The final cancellation table compares the two classifiers on the same test flights using the settings selected on the validation set.